In [1]:
import os

# Quay về thư mục gốc của Kaggle
%cd /kaggle/working

if not os.path.exists('/kaggle/working/OW-DETR'):
    !git clone https://github.com/nta2112/OW-DETR.git /kaggle/working/OW-DETR
else:
    print("Thư mục OW-DETR đã tồn tại. Đang đồng bộ cập nhật mới nhất từ GitHub...")
    %cd /kaggle/working/OW-DETR
    !git pull origin main

# Chuyển hẳn vào thư mục project
%cd /kaggle/working/OW-DETR


/kaggle/working
Cloning into '/kaggle/working/OW-DETR'...
remote: Enumerating objects: 344, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 344 (delta 179), reused 284 (delta 119), pack-reused 0 (from 0)
Receiving objects: 100% (344/344), 1.68 MiB | 8.66 MiB/s, done.
Resolving deltas: 100% (179/179), done.
/kaggle/working/OW-DETR


In [2]:
%cd /kaggle/working/OW-DETR

# 1. Cài đặt các thư viện
!pip install -r requirements.txt

# 2. Tải mô hình xương sống DINO ResNet-50
!mkdir -p models
!wget -N https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth -O models/dino_resnet50_pretrain.pth

# 3. Biên dịch toán tử CUDA
%cd models/ops
!sh make.sh
# Kiểm tra biên dịch (Nếu in ra toàn bộ True là thành công)
!python test.py

%cd /kaggle/working/OW-DETR


/kaggle/working/OW-DETR
for details.

--2026-07-12 09:17:07--  https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.88, 54.230.79.93, 54.230.79.53, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.88|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 94345885 (90M) [application/zip]
Saving to: ‘models/dino_resnet50_pretrain.pth’

models/dino_resnet5 100%[===================>]  89.97M   287MB/s    in 0.3s    

2026-07-12 09:17:07 (287 MB/s) - ‘models/dino_resnet50_pretrain.pth’ saved [94345885/94345885]

/kaggle/working/OW-DETR/models/ops
running build
running build_py
creating build/lib.linux-x86_64-cpython-312/functions
copying functions/ms_deform_attn_func.py -> build/lib.linux-x86_64-cpython-312/functions
copying functions/__init__.py -> build/lib.linux-x86_64-cpython-312/functions
creating build/lib.linux-x86_64-cpython-312/module

In [3]:
%cd /kaggle/working/OW-DETR

!python tools/convert_ip102_to_voc.py \
    --coco_dir /kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations \
    --img_dir /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification \
    --classes_file /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classes.txt \
    --output_dir data/IP102

/kaggle/working/OW-DETR
Đã copy /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classes.txt sang data/IP102/VOC2007/classes.txt
Đang xử lý split: train từ train.json...
loading annotations into memory...
Done (t=0.28s)
creating index...
index created!
Đã xử lý 2000 ảnh...
Đã xử lý 4000 ảnh...
Đã xử lý 6000 ảnh...
Đã xử lý 8000 ảnh...
Đã xử lý 10000 ảnh...
Đã xử lý 12000 ảnh...
Đã xử lý 14000 ảnh...
Đã xử lý 16000 ảnh...
Đã xử lý 18000 ảnh...
Đã xử lý 20000 ảnh...
Đã xử lý 22000 ảnh...
Đã xử lý 24000 ảnh...
Đã xử lý 26000 ảnh...
Đã xử lý 28000 ảnh...
Đã xử lý 30000 ảnh...
Đã xử lý 32000 ảnh...
Đã xử lý 34000 ảnh...
Đã xử lý 36000 ảnh...
Đã xử lý 38000 ảnh...
Đã xử lý 40000 ảnh...
Đã xử lý 42000 ảnh...
Đã xử lý 44000 ảnh...
Đã hoàn thành split: train.
Đang xử lý split: val từ val.json...
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Đã xử lý 2000 ảnh...
Đã xử lý 4000 ảnh...
Đã xử lý 6000 ảnh...
Đã hoàn thành split: val.
Đang xử lý split: test từ test.jso

In [4]:
%cd /kaggle/working/OW-DETR

from pycocotools.coco import COCO
import os
import random

json_path = "/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json"
coco = COCO(json_path)

# Định nghĩa chia lớp theo yêu cầu mới:
# Task 1: 27 lớp (0 - 26)
# Task 2: 25 lớp (27 - 51)
# Task 3: 25 lớp (52 - 76)
# Task 4: 25 lớp (77 - 101)
task_splits = {
    't1': list(range(0, 27)),
    't2': list(range(27, 52)),
    't3': list(range(52, 77)),
    't4': list(range(77, 102))
}

# 1. Tạo tập train cho các Task
train_images = {}
for task_name, classes in task_splits.items():
    task_img_ids = set()
    for ann_id, ann in coco.anns.items():
        if ann['category_id'] in classes:
            task_img_ids.add(ann['image_id'])
    train_images[task_name] = sorted(list(task_img_ids))

# Hàm tạo tập exemplars (finetuning) lưu trữ thông tin cũ
def get_exemplars(coco, class_ids, num_images_per_class=20):
    exemplar_ids = set()
    for cid in class_ids:
        img_ids = [ann['image_id'] for ann in coco.anns.values() if ann['category_id'] == cid]
        selected = img_ids[:num_images_per_class]
        exemplar_ids.update(selected)
    return exemplar_ids

# 2. Tạo tập Finetuning (ft)
ft_images = {}
ft_images['t2'] = sorted(list(set(train_images['t2']).union(get_exemplars(coco, task_splits['t1']))))
ft_images['t3'] = sorted(list(set(train_images['t3']).union(get_exemplars(coco, task_splits['t1'] + task_splits['t2']))))
ft_images['t4'] = sorted(list(set(train_images['t4']).union(get_exemplars(coco, task_splits['t1'] + task_splits['t2'] + task_splits['t3']))))

# 3. Ghi tất cả các file splits ra đĩa
output_dir = "data/IP102/VOC2007/ImageSets/Main"
os.makedirs(output_dir, exist_ok=True)

splits = {
    't1_train': train_images['t1'],
    't2_train': train_images['t2'],
    't3_train': train_images['t3'],
    't4_train': train_images['t4'],
    't2_ft': ft_images['t2'],
    't3_ft': ft_images['t3'],
    't4_ft': ft_images['t4']
}

for name, img_ids in splits.items():
    path = os.path.join(output_dir, f"{name}.txt")
    with open(path, "w") as f:
        for img_id in img_ids:
            f.write(f"{img_id:06d}\n")
    print(f"Đã tạo {name}.txt với {len(img_ids)} ảnh.")


/kaggle/working/OW-DETR
loading annotations into memory...
Done (t=0.18s)
creating index...
index created!
Đã tạo t1_train.txt với 13447 ảnh.
Đã tạo t2_train.txt với 9746 ảnh.
Đã tạo t3_train.txt với 12742 ảnh.
Đã tạo t4_train.txt với 9160 ảnh.
Đã tạo t2_ft.txt với 10286 ảnh.
Đã tạo t3_ft.txt với 13782 ảnh.
Đã tạo t4_ft.txt với 10700 ảnh.


In [5]:
# # Cell 5: Chuyển đổi, ánh xạ và cắt query_embed tương thích --num_queries 100
# import torch
# import os

# # Đường dẫn file trên Kaggle của bạn
# input_path = "/kaggle/input/models/nta212/deformable/pytorch/default/1/pytorch_model.bin" 
# converted_path = "/kaggle/working/OW-DETR/models/deformable_detr_coco_converted.pth"

# print("-> Đang tiến hành đọc và ánh xạ trọng số tương thích...")
# os.makedirs("/kaggle/working/OW-DETR/models", exist_ok=True)

# if not os.path.exists(converted_path):
#     if not os.path.exists(input_path):
#         raise FileNotFoundError(f"Không tìm thấy file tại {input_path}. Vui lòng kiểm tra lại cột Input bên phải Kaggle.")
        
#     state_dict = torch.load(input_path, map_location="cpu")
#     new_state_dict = {}
    
#     for k, v in state_dict.items():
#         # 1. Bỏ qua class_embed do lệch số lượng class
#         if "class_embed." in k:
#             continue
            
#         new_k = k
#         # 2. Ánh xạ phần Backbone
#         if k.startswith("model.backbone.conv_encoder.model."):
#             new_k = k.replace("model.backbone.conv_encoder.model.", "backbone.0.body.")
#         # 3. Ánh xạ phần Transformer Encoder
#         elif k.startswith("model.encoder."):
#             new_k = k.replace("model.encoder.", "transformer.encoder.")
#         # 4. Ánh xạ phần Transformer Decoder
#         elif k.startswith("model.decoder."):
#             new_k = k.replace("model.decoder.", "transformer.decoder.")
#         # 5. Ánh xạ Level Embed và Reference Points
#         elif k == "model.level_embed":
#             new_k = "transformer.level_embed"
#         elif k.startswith("model.reference_points."):
#             new_k = k.replace("model.reference_points.", "transformer.reference_points.")
#         # 6. Ánh xạ và CẮT BỚT Query Embedding (từ 300 xuống 100 queries)
#         elif k == "model.query_position_embeddings.weight":
#             new_k = "query_embed.weight"
#             if v.shape[0] > 100:
#                 print(f"  [Đã cắt] query_embed.weight từ {list(v.shape)} xuống [100, 512]")
#                 v = v[:100, :]  # Chỉ lấy 100 query đầu tiên để khớp với mô hình sâu bệnh
#         # 7. Các phần còn lại bỏ tiền tố "model."
#         elif k.startswith("model."):
#             new_k = k[6:]
            
#         new_state_dict[new_k] = v

#     # Lưu lại checkpoint mới đã xử lý hoàn toàn
#     torch.save({"model": new_state_dict, "epoch": -1}, converted_path)
#     print(f" Hoàn thành! Đã lưu checkpoint tương thích 100% tại: {converted_path}")
# else:
#     print("-> Checkpoint tương thích đã tồn tại, bỏ qua bước chuyển đổi.")

# Task 1 ? quên r

In [6]:
# %cd /kaggle/working/OW-DETR

# !torchrun --nproc_per_node=2 main_open_world.py \
#     --output_dir '/kaggle/working/exps/ip102_t1' \
#     --dataset owod \
#     --num_queries 100 \
#     --batch_size 2 \
#     --PREV_INTRODUCED_CLS 0 \
#     --CUR_INTRODUCED_CLS 27 \
#     --data_root 'data/IP102' \
#     --train_set 't1_train' \
#     --val_set 'val' \
#     --test_set 'test' \
#     --num_classes 103 \
#     --unmatched_boxes \
#     --top_unk 5 \
#     --featdim 1024 \
#     --NC_branch \
#     --backbone 'dino_resnet50' \
#     --pretrain 'models/deformable_detr_coco_converted.pth' \
#     --num_workers 4 \
#     --eval_every 1 \
#     --cache_mode \
#     --epochs 1


In [7]:
# %cd /kaggle/working/OW-DETR

# !torchrun --nproc_per_node=2 main_open_world.py \
#     --output_dir '/kaggle/working/exps/ip102_t1' \
#     --dataset owod \
#     --num_queries 100 \
#     --batch_size 10 \
#     --PREV_INTRODUCED_CLS 0 \
#     --CUR_INTRODUCED_CLS 27 \
#     --data_root 'data/IP102' \
#     --train_set 't1_train' \
#     --val_set 'val' \
#     --test_set 'test' \
#     --num_classes 103 \
#     --unmatched_boxes \
#     --top_unk 5 \
#     --featdim 1024 \
#     --NC_branch \
#     --backbone 'dino_resnet50' \
#     --pretrain 'models/deformable_detr_coco_converted.pth' \
#     --num_workers 4 \
#     --eval_every 1 \
#     --cache_mode \
#     --epochs 1

# Task 2

In [8]:
%cd /kaggle/working/OW-DETR

!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t2' \
    --dataset owod \
    --dec_layers 6 \
    --num_queries 100 \
    --batch_size 2 \
    --lr 0.0002 \
    --PREV_INTRODUCED_CLS 27 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't2_ft' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --pretrain '/kaggle/input/models/nta212/task-1/pytorch/default/1/best_checkpoint.pth' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --nc_epoch 2 \
    --filter_pct 0.5 \
    --epochs 6

/kaggle/working/OW-DETR
W0712 09:27:57.381000 121 torch/distributed/run.py:852] 
W0712 09:27:57.381000 121 torch/distributed/run.py:852] *****************************************
W0712 09:27:57.381000 121 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0712 09:27:57.381000 121 torch/distributed/run.py:852] *****************************************
('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'diningtable', 'pottedplant', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'bed', 'toilet', 'sofa', 'frisbee', 'skis', 'snowboard', 'spo

In [9]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --dec_layers 6 \
    --num_queries 100 \
    --batch_size 10 \
    --PREV_INTRODUCED_CLS 27 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't2_ft' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t2/best_checkpoint.pth' \
    --eval

W0712 10:40:12.794000 598 torch/distributed/run.py:852] 
W0712 10:40:12.794000 598 torch/distributed/run.py:852] *****************************************
W0712 10:40:12.794000 598 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0712 10:40:12.794000 598 torch/distributed/run.py:852] *****************************************
('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'diningtable', 'pottedplant', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'bed', 'toilet', 'sofa', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'base